# 01 · Spatial contracts and a tiny city

**Question:** Can we preserve coordinates, units and feature identities across a twin pipeline?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
All city geometry, residents, sensor observations and parameters are **synthetic educational fixtures**.
The longitude/latitude anchor is near Gaborone, but these are not mapped Gaborone assets or a validated city twin.

**Astra experiment:** Ask Astra to detect a swapped coordinate pair, explain local metres versus geographic degrees, and propose validation rules.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## Coordinate frame and editable inputs
GeoJSON uses `[longitude, latitude]`. Calculations use a local equirectangular approximation in metres over a 2 km area.
This is not an EPSG projected CRS; use a vetted projection library for larger areas or survey work.
The site-specific bounding box is deliberately stronger than global coordinate range checks.


In [ ]:
c = city()
population_total = 1200
lon, lat = lonlat(*c['xy'].T)
x2,y2 = local_xy(lon,lat)
error_m = np.max(np.hypot(x2-c['xy'][:,0],y2-c['xy'][:,1]))
assert error_m < 1e-6
assert ((lon>25.8)&(lon<26.1)&(lat>-24.8)&(lat<-24.5)).all()
features_geo = []
for i,(x,y) in enumerate(c['xy']):
    w,d=c['width'][i]/2,c['depth'][i]/2
    ring=[list(map(float,lonlat(a,b))) for a,b in [(x-w,y-d),(x+w,y-d),(x+w,y+d),(x-w,y+d),(x-w,y-d)]]
    features_geo.append(dict(type='Feature',id=f'b{i:03d}',properties=dict(synthetic=True,floors=int(c['floors'][i])),geometry=dict(type='Polygon',coordinates=[ring])))
geojson=dict(type='FeatureCollection',features=features_geo)
assert len({f['id'] for f in features_geo})==len(features_geo)
print('Buildings:',len(features_geo),'| round-trip error (m):',error_m)


In [ ]:
fig,ax=plt.subplots()
for f in features_geo:
    ring=np.array(f['geometry']['coordinates'][0]); xx,yy=local_xy(*ring.T)
    ax.fill(xx,yy,color='#087f8c',alpha=.55)
map_axes(ax,'Synthetic urban footprint inventory'); plt.show()
export_json('01_buildings.geojson',geojson)


## Deliberately break the contract
A pair can pass world-wide latitude/longitude bounds and still be in the wrong country.
Spatial plausibility needs study-area bounds and provenance, not only valid JSON.


In [ ]:
good=[float(lon[0]),float(lat[0])]; bad=good[::-1]
def in_study_area(pair):
    return 25.8<pair[0]<26.1 and -24.8<pair[1]<-24.5
assert in_study_area(good) and not in_study_area(bad)
print('Astra prompt: Audit these GeoJSON coordinate pairs near Gaborone:',good,bad,
      'Explain which checks fail. Do not invent a surveyed coordinate.')


## Teaching lab: Audit coordinates
**Before using AI:** write a prediction, run the numerical baseline, and explain one surprising result.

**Astra evidence:** send only 01_buildings.geojson plus the two coordinate pairs printed above. Use a fresh conversation and record the exact model,
date, interface, tools and prompt. A task with a supplied result tests interpretation, not independent calculation.

**Copyable Astra prompt:**
> Identify coordinate order, frame and units. Explain which pair violates the study bounds and why global bounds alone are insufficient.

**Copyable Codex task:**
> Add a GeoJSON audit returning feature IDs for unclosed rings, duplicate IDs and points outside the study bounds. Edit the authoring source in `scripts/make_notebooks.py` (or
> `scripts/teaching_labs.py` for labs 11–13), regenerate the notebooks and preserve browser compatibility.
> Use NumPy, Matplotlib and the standard library. Include a small reference case that can be checked by hand.

**Acceptance evidence:** An intentionally reversed coordinate pair fails; unchanged valid features pass; no automatic axis swapping.

**Share back:** save your original prediction, changed parameter, result, model response and one limitation.
Download the edited notebook and exports. If you have no model access, exchange explanations with a partner
and evaluate them with the same criteria; do not label that activity as an Astra trial.


## Extend with Codex or Astra
Replace the synthetic inventory with a small, licensed OSM/Overture extract. Preserve source IDs, retrieval date, attribution and CRS. Add geometry validity checks before computing areas.

Keep the baseline seed and tests fixed while changing one assumption. Record the model name,
date, exact prompt, response and error metrics. Generated code must pass the same checks.
Download files in `exports/` and your edited notebook; browser storage does not commit changes to GitHub.
See [the project guide](https://github.com/jltobias/JupyterLite-GPT-6-Astra-Geospatial/blob/main/docs/GUIDE.md)
for capabilities, evidence, sources and the route to real data.
